# 🧩 Lesson 13: The Full RAG Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/13_full_rag_pipeline.ipynb)

> **Twelve lessons, twelve pieces. Now we connect them into one system you can call with a single function.**

---

## 🎯 Goal
By the end of this lesson you have a working RAG system on the **real handbook PDF**: `rag_answer(question)` returns an answer, the sources it used, and a check of its citations.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~30 minutes |
| 📚 **Prerequisites** | Lessons [02](02_loading.ipynb) to [12](12_generation.ipynb) |
| 🔑 **API key** | Required (OpenRouter) |
| 🛠 **Libraries** | `pdfplumber`, `langchain-text-splitters`, `chromadb`, `sentence-transformers`, `langchain`, `langchain-openrouter` |

In [1]:
#!pip install -q pdfplumber langchain-text-splitters chromadb sentence-transformers numpy "langchain>=1" langchain-openrouter

## 🗺️ The whole pipeline

| Stage | Lesson | Function we write |
|---|---|---|
| Load + parse into sections | 02, 03 | `parse_sections` |
| Clean | 04 | `clean_text` |
| Chunk + metadata | 05, 06 | `make_chunks` |
| Embed + store | 07, 09 | `build_index` |
| Retrieve | 11 | `retrieve` |
| Generate + check | 12 | `rag_answer` |

**Part 1 (this half): ingestion**, run once per document. **Part 2: question answering**, run for every question.

All the settings live in **one cell**, so changing a model or a chunk size is a one-line edit. (In the final repo these functions move into a `src/` folder and notebooks just import them.)

In [2]:
import re
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path

import chromadb
import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_URL = "https://raw.githubusercontent.com/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/main/data/pdf1.pdf"
EMBED_MODEL = "intfloat/multilingual-e5-small"
CHAT_MODEL = "openai/gpt-4o-mini"
MAX_CHARS = 600
TOP_K = 4
COLLECTION_NAME = "handbook_pipeline"

SYSTEM_PROMPT = (
    "You answer questions about a company handbook. "
    "Use ONLY the context provided. "
    "Cite the ids of the chunks you used in square brackets, like [pdf1-2-0]. "
    "If the context does not contain the answer, reply exactly: I don't know based on the handbook."
)

In [3]:
def find_pdf():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        for relative in ("data/pdf1.pdf", "source/pdf1.pdf"):
            candidate = folder / relative
            if candidate.exists():
                return candidate
    pdf_path = Path("data/pdf1.pdf")
    pdf_path.parent.mkdir(exist_ok=True)
    try:
        with urllib.request.urlopen(PDF_URL, timeout=15) as response:
            pdf_path.write_bytes(response.read())
    except OSError as error:
        raise FileNotFoundError(
            f"pdf1.pdf not found in 'data' or 'source' of {Path.cwd()} or any parent folder, and the download failed ({error}). "
            "Copy pdf1.pdf into a 'data' folder of your project."
        )
    return pdf_path

def parse_sections(pdf_path):
    lines = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages):
            for line in page.extract_text_lines():
                size = round(max(char["size"] for char in line["chars"]), 1)
                lines.append({"text": line["text"], "size": size, "page": page_number})
    body_size = Counter(line["size"] for line in lines).most_common(1)[0][0]
    sections, current = [], None
    for line in lines:
        if line["size"] > body_size:
            current = {"title": line["text"], "page": line["page"], "text": []}
            sections.append(current)
        elif current is not None:
            current["text"].append(line["text"])
    return [section for section in sections if section["text"]]

def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("(cid:127)", "\n•")
    text = re.sub(r"Full[n■]time", "Full-time", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

def make_chunks(sections, source):
    splitter = RecursiveCharacterTextSplitter(chunk_size=MAX_CHARS, chunk_overlap=60)
    chunks = []
    for index, section in enumerate(sections):
        text = clean_text(" ".join(section["text"]))
        parts = [text] if len(text) <= MAX_CHARS else splitter.split_text(text)
        for part_number, part in enumerate(parts):
            chunks.append({
                "id": f"{Path(source).stem}-{index}-{part_number}",
                "text": part,
                "metadata": {"source": source, "page": section["page"], "section": section["title"], "part": part_number},
            })
    return chunks

In [4]:
pdf_path = find_pdf()
sections = parse_sections(pdf_path)
chunks = make_chunks(sections, pdf_path.name)

for chunk in chunks:
    print(chunk["id"], "|", len(chunk["text"]), "chars |", chunk["metadata"]["section"])

print()
print(chunks[2]["text"])

pdf1-0-0 | 427 chars | Non-Disclosure Agreement (NDA) Policy
pdf1-1-0 | 360 chars | Workplace Conduct Policy
pdf1-2-0 | 558 chars | Paid Time Off (PTO) Policy
pdf1-3-0 | 462 chars | Travel & Expense Policy

Full-time employees accrue PTO according to the following schedule:
• 0–1 years of service: 10 days per year (0.833 days per month)
• 1–3 years of service: 15 days per year (1.25 days per month)
• 3+ years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal needs, or illness. Requests should be submitted in advance through the HR system unless related to an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended absences exceeding 5 consecutive business days require manager approval.


## 🏗️ Part 2: index and answer

Ingestion gave us `chunks`. Now:

1. **`build_index`**: embed the chunks and store them in Chroma. We **rebuild the collection from scratch** each time, so chunks from an older run can't stay behind.
2. **`retrieve`**: find the nearest chunks for a question.
3. **`rag_answer`**: retrieve, ask the model, and return **everything useful in one dictionary**: the answer, whether it refused, which chunks it cited, any invented ids, and the sources with their page.

In [5]:
from getpass import getpass
import os
from sentence_transformers import SentenceTransformer
from langchain.chat_models import init_chat_model

embedder = SentenceTransformer(EMBED_MODEL)

if "OPENROUTER_API_KEY" not in os.environ:
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

model = init_chat_model(
    model=CHAT_MODEL,
    model_provider="openrouter",
    temperature=0,
)

client = chromadb.PersistentClient(path="chroma_db")
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    configuration={"hnsw": {"space": "cosine"}},
)

def build_index(chunks):
    vectors = embedder.encode(["passage: " + chunk["text"] for chunk in chunks], normalize_embeddings=True)
    collection.upsert(
        ids=[chunk["id"] for chunk in chunks],
        embeddings=vectors.tolist(),
        documents=[chunk["text"] for chunk in chunks],
        metadatas=[chunk["metadata"] for chunk in chunks],
    )
    return collection.count()

print("stored:", build_index(chunks))

d:\AI Source\Rag_torurial\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8993.96it/s]


stored: 4


## 🎁 One function that returns everything

`rag_answer(question)` returns a **dictionary**, not just text. Why? Because the next steps (a user interface, an evaluation, logging) need more than the answer: they need to know *what was retrieved*, *what was cited* and *whether the model refused*.

| Key | Meaning |
|---|---|
| `answer` | the model's text |
| `refused` | `True` if it said "I don't know" |
| `cited` | chunk ids it cited that **were** retrieved |
| `invented` | ids it cited that were **not** retrieved (should be empty) |
| `sources` | retrieved chunks with section, page (counted from **1**) and similarity |

In [6]:
def retrieve(question, top_k=TOP_K, where=None):
    question_vector = embedder.encode(["query: " + question], normalize_embeddings=True)[0]
    result = collection.query(
        query_embeddings=[question_vector.tolist()],
        n_results=top_k,
        where=where,
    )
    return [
        {"id": chunk_id, "section": metadata["section"], "page": metadata["page"], "similarity": 1 - distance, "text": text}
        for chunk_id, metadata, distance, text in zip(
            result["ids"][0], result["metadatas"][0], result["distances"][0], result["documents"][0]
        )
    ]

def format_context(items):
    return "\n".join(f'[{item["id"]}] {item["text"]}' for item in items)

def rag_answer(question, top_k=TOP_K, where=None):
    items = retrieve(question, top_k=top_k, where=where)
    respone = model.invoke(
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{format_context(items)}\n\nQuestion: {question}"},
        ]
    )
    text = respone.content
    cited = list(dict.fromkeys(re.findall(r"\[([\w-]+)\]", text)))
    retrieved_ids = [item["id"] for item in items]
    return {
        "question": question,
        "answer": text,
        "refused": "I don't know" in text,
        "cited": [c for c in cited if c in retrieved_ids],
        "invented": [c for c in cited if c not in retrieved_ids],
        "sources": [
            {"id": item["id"], "section": item["section"], "page": item["page"] + 1, "similarity": round(item["similarity"], 3)}
            for item in items
        ],
    }

In [7]:
questions = [
    "How many PTO days do I get after five years, and how do I request them?",
    "Can I carry over unused PTO days?",
    "What is the dental insurance policy?",
]

for question in questions:
    result = rag_answer(question, top_k=2)
    print("Q:", result["question"])
    print("A:", result["answer"])
    print("   cited   :", result["cited"], "| invented:", result["invented"], "| refused:", result["refused"])
    print("   sources :", [(source["section"][:22], f'page {source["page"]}', source["similarity"]) for source in result["sources"]])
    print()

Q: How many PTO days do I get after five years, and how do I request them?
A: After five years, you would accrue 20 PTO days per year (1.67 days per month) [pdf1-2-0]. To request PTO, you should submit your request in advance through the HR system unless it is related to an emergency [pdf1-2-0].
   cited   : ['pdf1-2-0'] | invented: [] | refused: False
   sources : [('Paid Time Off (PTO) Po', 'page 1', 0.894), ('Travel & Expense Polic', 'page 1', 0.831)]

Q: Can I carry over unused PTO days?
A: Yes, you may carry over up to 5 unused PTO days per calendar year [pdf1-2-0].
   cited   : ['pdf1-2-0'] | invented: [] | refused: False
   sources : [('Paid Time Off (PTO) Po', 'page 1', 0.892), ('Travel & Expense Polic', 'page 1', 0.809)]

Q: What is the dental insurance policy?
A: I don't know based on the handbook.
   cited   : [] | invented: [] | refused: True
   sources : [('Paid Time Off (PTO) Po', 'page 1', 0.773), ('Non-Disclosure Agreeme', 'page 1', 0.769)]



### 👀 What we saw

| Question | Answer | Cited | Refused |
|---|---|---|---|
| PTO after five years + how to request | 20 days/year (1.67/month); request through the HR system | `pdf1-2-0` | no |
| Carry over unused PTO? | yes, up to 5 days per calendar year | `pdf1-2-0` | no |
| Dental insurance? | "I don't know based on the handbook." | none | **yes** |

- Both answerable questions were answered **correctly** (we checked them against the PDF), and cited a chunk that was really retrieved (`invented` was empty every time).
- **Scores** matched the pattern from earlier lessons: the relevant chunk scored about **0.89**, while the unanswerable question's best chunks scored about **0.77**.
- In Lesson 11, with tiny chunks, the wrong PTO tier ranked first. Here the three tiers sit **in one chunk**, and the question was answered correctly. Chunking and retrieval work together.

## ⚠️ Honest limits of this pipeline

- **Four chunks, one document.** With `top_k=2`, the second source is a filler (Travel, similarity 0.81 to 0.83). Real retrieval only gets interesting with many chunks.
- **Three questions prove nothing.** We have *no measurement* yet. The Evaluation lessons (14 to 17) turn "it looked right" into numbers.
- **The "I don't know" check is text-based.** `refused` looks for the exact phrase. A differently worded refusal would be missed.
- **No similarity cut-off.** We rely on the prompt to refuse. As Lesson 11 showed, any threshold must be measured, not guessed.
- **It re-reads and re-embeds the PDF every time you run ingestion.** Fine for one small file, wasteful for a big library (Lesson 32 covers updating an index).
- **Every question calls the paid chat model.** Cost per question was small in our tests, but it is not zero.
- **Models can change.** The same code may behave differently with another chat model or embedding model. Re-test after any change.

## 📌 Summary

- A RAG system is a **pipeline of small, testable functions**: `parse_sections`, `clean_text`, `make_chunks`, `build_index`, `retrieve`, `rag_answer`.
- **Settings live in one cell**, so changing a model, a chunk size or `top_k` is a one-line edit.
- `rag_answer` returns a **dictionary** (answer, refused, cited, invented, sources), not just text, so later steps can evaluate and display it.
- Chunks keep their **metadata** all the way to the answer, so we can show section and page.
- The prompt makes the model **cite** and **refuse**. Code **checks** the citations.
- We have a working RAG on a real PDF, but we have **not measured** how good it is. That is the next module.

| Lessons | What they built |
|---|---|
| 00 to 01 | the idea, and why RAG |
| 02 to 06 | ingestion: load, parse, clean, metadata, chunk |
| 07 to 10 | embeddings, model choice, vector DB, indexing |
| 11 to 12 | retrieval and generation |
| **13** | **everything, wired together** |

## ✍️ Exercise (15 minutes)

1. Write **five** questions about the handbook: three it can answer, two it cannot.
2. Run `rag_answer` on each. For every answer, check **by hand** against the PDF: correct, wrong, or refused.
3. Make a small table: question, correct?, refused?, best similarity.

**Success:** you can state how many of your five questions the system handled correctly, and you can point to one place where you'd like to see it improve.

💡 Solution: `solutions/13_full_rag_pipeline.ipynb` (try on your own first)

## 🔥 Challenge (optional)

Our embedding model is multilingual, but the handbook is in English. Ask in **Persian**:

`بعد از پنج سال چند روز مرخصی دارم؟`

1. **Predict first:** will it retrieve the right chunk? Will the model answer? In which language?
2. Run it. Were you right?
3. What does this tell you about mixing languages in a RAG system?

## 📖 Next

We have a pipeline, but we judge it by eye. In the next module we **measure** it: build a test set, compute retrieval metrics, and score the answers.

**➡️ Next lesson:** [14 Evaluation Basics](14_evaluation_basics.ipynb)